# 워밍업 — 비슷한 문장 찾기 🗺️

웹 챕터에서 본 두 실험을 이번엔 **직접 코드로** 만져 봅니다.

> **노트북 사용법**: 각 셀(회색 상자)을 클릭하고 `Shift + Enter`를 누르세요. 위에서 아래로 반복하면 끝.
> 코드를 다 이해하지 못해도 괜찮아요. 실행해서 결과를 눈으로 보는 게 목적입니다.

**이 노트북의 목표**: 7부 전체의 핵심 문장을 코드로 확인한다.
**"의미는 위치다: 비슷함은 가까움으로 잴 수 있다."**

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼을 눌러 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('sample')

## 1. 소박한 방법: 글자 겹침으로 유사도 재기

컴퓨터의 가장 단순한 유사도: **겹치는 단어 수 세기**. 어디까지 통하는지 봅시다.

In [ ]:
QUERY = '회식 장소 어디로 정했어?'
CANDS = {
    'A': '저녁 모임 위치는 결정됐나요?',   # 뜻 같음 (사람의 정답)
    'B': '회식 장소 예약은 제가 할게요.',  # 단어 겹침, 뜻 다름 (질문 아님)
    'C': '주말 날씨가 참 좋네요.',        # 완전 무관
}

def overlap_score(a, b):
    """겹치는 단어 수 — 컴퓨터의 가장 소박한 유사도."""
    return len(set(a.split()) & set(b.split()))

print(f'질문: {QUERY}\n')
scores = {}
for k, s in CANDS.items():
    scores[k] = overlap_score(QUERY, s)
    print(f'  {k} ({s})  →  겹침 {scores[k]}개')

winner = max(scores, key=scores.get)
print(f'\n🤖 글자 겹침의 1등: {winner}  ← 사람의 정답은 A인데!')
assert winner == 'B', '글자 겹침은 B(단어만 겹침)를 1등으로 뽑아야 — 맹점의 재현'
assert scores['A'] == scores['C'] == 0, '정답 A가 무관한 C와 동점 꼴찌여야 — 맹점의 핵심'

**관찰**: 단어가 같은 것과 뜻이 같은 것은 다른 층위입니다. 키워드 검색의 맹점이죠.

## 2. 의미 지도: 뜻을 좌표로

해법: 뜻을 **좌표**로 바꿔서(임베딩), "비슷한가?"를 "**가까운가?**"로 계산합니다.
여기서는 미리 준비한 **2차원 toy 임베딩**을 씁니다 (진짜는 수백 차원, 4장에서).

In [ ]:
import math

# 미리 준비한 단어 지도 (2차원 toy 임베딩)
WORD_MAP = {
    '강아지': (2.0, 7.5), '고양이': (2.8, 8.2), '햄스터': (2.4, 6.8),
    '버스': (7.0, 3.0), '자동차': (7.8, 3.7), '기차': (7.3, 2.2),
    '사과': (7.4, 8.2), '바나나': (8.2, 7.5),
}

def distance(p, q):
    """두 점 사이의 거리 (1부 1~2장의 좌표와 피타고라스)."""
    return math.sqrt((p[0] - q[0]) ** 2 + (p[1] - q[1]) ** 2)

# 새 단어 '토끼'가 지도에 들어왔다 — 어디와 가까울까?
rabbit = (2.2, 8.0)
ranked = sorted(WORD_MAP, key=lambda w: distance(rabbit, WORD_MAP[w]))
print('🐰 토끼와 가까운 순서:')
for w in ranked[:4]:
    print(f'   {w:6s} 거리 {distance(rabbit, WORD_MAP[w]):.2f}')

assert set(ranked[:3]) == {'강아지', '고양이', '햄스터'}, '토끼의 이웃은 동물들이어야!'
print('\n✔ 확인: 가까움 = 비슷함. 지도가 뜻을 알고 있다')

## 3. 문장 버전: 의미 검색 흉내

문장에도 좌표를 주면(문장 임베딩), 실험 1에서 실패했던 문제가 풀립니다.

In [ ]:
# 미리 준비한 문장 좌표 (뜻이 같으면 가깝게 — 9장에서 모델이 이걸 배우는 법을 다룬다)
SENT_MAP = {
    '회식 장소 어디로 정했어?':     (5.0, 5.0),
    '저녁 모임 위치는 결정됐나요?': (5.3, 5.2),  # 뜻 같음 → 가까이
    '회식 장소 예약은 제가 할게요.': (6.8, 4.1),  # 단어 겹쳐도 뜻 다름 → 멀리
    '주말 날씨가 참 좋네요.':       (1.5, 1.0),  # 무관 → 아주 멀리
}

q = SENT_MAP[QUERY]
print(f'질문: {QUERY}\n')
results = sorted(
    (s for s in SENT_MAP if s != QUERY),
    key=lambda s: distance(q, SENT_MAP[s]),
)
for s in results:
    print(f'   거리 {distance(q, SENT_MAP[s]):5.2f}  {s}')

assert results[0] == '저녁 모임 위치는 결정됐나요?', '의미 지도에서는 뜻이 같은 문장이 1등이어야!'
print('\n✔ 확인: 글자 겹침이 놓친 정답을, 거리는 찾아냈다 (의미 검색의 원리)')

## 4. 관찰 정리 + 🤖 AI 연결

- 글자 겹침은 **단어**를 재고, 거리는 **뜻**을 잰다. 단, 뜻이 좌표로 잘 배치되어 있을 때만.
- 좌표와 거리 계산은 **1부 1~2장** 그대로입니다. 새 수학이 아니라 아는 수학의 새 무대.
- 남은 질문들이 곧 이 과정의 목차입니다: 좌표는 누가 정하나(9장), 거리 말고 더 좋은 자는
  없나(2장, 코사인), 수백 차원에서도 이 감각이 통하나(4장), 문서 수만 개에서 1등을
  빨리 찾으려면(6장), 이걸로 회사 챗봇을 만들려면(8·10장).

**다음 챕터**: 1장(검색의 진화와 RAG). 이 지도로 무엇을 만들 수 있는지 전체 그림을 폅니다.

> 💡 궁금한 코드는 그 셀을 실행한 다음 셀에서 `from tutor import explain` 후 `explain('sample')`.